In [1]:
from tce.data import (
    download_data, 
    extract_archive,
    save_data,
    load_data,
    join_data, 
    clean_text, 
    process_split
)
from tce.utils import load_json
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

paths_path = PROJECT_ROOT / 'config/paths.json'
paths = load_json(paths_path)

raw_dir = PROJECT_ROOT / paths['data_raw_dir']
data_url = paths['data_url']
arch_path = raw_dir / paths['aclImdb_v1']
download_data(data_url, arch_path)

ext_path = raw_dir / paths['aclImdb']
extract_archive(arch_path, raw_dir, ext_path)

The data is dowloaded and saved to the directory ..\\data\\raw.

In [3]:
raw_train_dir = ext_path / paths['train']
raw_test_dir = ext_path / paths['test']

proc_dir = PROJECT_ROOT / paths['data_processed_dir']

train_texts_path = raw_dir / paths['raw_train']
test_texts_path = raw_dir / paths['raw_test']

train_labels_path = proc_dir / paths['train_labels']
test_labels_path = proc_dir / paths['test_labels']

train_cleaned_path = proc_dir / paths['processed_train']
test_cleaned_path = proc_dir / paths['processed_test']

# saving raw texts (train/test) into a single file
# saving labels
# cleaning raw texts and saving processed texts into a single file
process_split(raw_train_dir, train_texts_path, train_labels_path, train_cleaned_path)
process_split(raw_test_dir, test_texts_path, test_labels_path, test_cleaned_path)

Texts are gathered together to one list and save to directory ..\\data\\raw as test-texts.txt for testing set texts and train-texts.txt for training set.

In [4]:
X_train = load_data(train_texts_path)
X_test = load_data(test_texts_path)

y_train = load_data(train_labels_path)
y_test = load_data(test_labels_path)

In [5]:
print(f'Training set size: {len(X_train)}')
print(f'Test set size: {len(X_test)}')

Training set size: 25000
Test set size: 25000


In [6]:
print('Amount of neg class in training set:', sum([1 for i in y_train if i == 0]))
print('Amount of pos class in training set:', sum([1 for i in y_train if i == 1]))
print()
print('Amount of neg class in testing set:', sum([1 for i in y_test if i == 0]))
print('Amount of pos class in testing set:', sum([1 for i in y_test if i == 1]))

Amount of neg class in training set: 12500
Amount of pos class in training set: 12500

Amount of neg class in testing set: 12500
Amount of pos class in testing set: 12500


Training set size equals testing set size equals 25 000. Total 50 000 objects. 

Classes are well balanced: 50% of positive texts and 50% of negative ones.

In [7]:
X_train_cleaned = load_data(train_cleaned_path)
X_test_cleaned = load_data(test_cleaned_path)

The texts were cleaned manually in this notebook for the analysis by clean_texts function. However, then they are cleaned within a TfidfVectorizer object while training the model.

In [8]:
print('Average number of words in training set:', round(sum([len(text.split()) for text in X_train_cleaned]) / len(X_train_cleaned)))
print('Average number of words in test set:', round(sum([len(text.split()) for text in X_test_cleaned]) / len(X_test_cleaned)))
print('Average number of symbols in training set:', round(sum([len(text) for text in X_train_cleaned]) / len(X_train_cleaned)))
print('Average number of symbols in test set:', round(sum([len(text) for text in X_test_cleaned]) / len(X_test_cleaned)))

Average number of words in training set: 232
Average number of words in test set: 227
Average number of symbols in training set: 1259
Average number of symbols in test set: 1229


In [9]:
print('Maximum number of words in training set:', max([len(text.split()) for text in X_train_cleaned]))
print('Maximum number of words in test set:', max([len(text.split()) for text in X_test_cleaned]))
print('Maximum number of symbols in training set:', max([len(text) for text in X_train_cleaned]))
print('Maximum number of symbols in test set:', max([len(text) for text in X_test_cleaned]))

Maximum number of words in training set: 2460
Maximum number of words in test set: 2234
Maximum number of symbols in training set: 13281
Maximum number of symbols in test set: 12259


In [10]:
print('Minimum number of words in training set:', min([len(text.split()) for text in X_train_cleaned]))
print('Minimum number of words in test set:', min([len(text.split()) for text in X_test_cleaned]))
print('Minimum number of symbols in training set:', min([len(text) for text in X_train_cleaned]))
print('Minimum number of symbols in test set:', min([len(text) for text in X_test_cleaned]))

Minimum number of words in training set: 10
Minimum number of words in test set: 6
Minimum number of symbols in training set: 51
Minimum number of symbols in test set: 30


There is a big range of lengths of comments. However, according to the average values, most of the comments are not so big. Also, it seems that training set's texts are longer than testing set's ones.

In [11]:
freq_train_dict = {}
for text in X_train_cleaned:
    for word in text.split():
        freq_train_dict[word] = freq_train_dict.get(word, 0) + 1

freq_train_list = []
for word, freq in freq_train_dict.items():
    freq_train_list.append((word, freq))

freq_train_list = sorted(freq_train_list, key=lambda x: x[1], reverse=True)

print('The most frequent words:')
i = 0
while i <= 10:
    print(f'{freq_train_list[i][0]}: {freq_train_list[i][1]}')
    i += 1

print('\nThe least frequent words:')
i -= 1
while i > 0:
    print(f'{freq_train_list[-i][0]}: {freq_train_list[-i][1]}')
    i -= 1

print('\nWords count:', len(freq_train_list))
print('\nCount of word with frequency = 1:', [x[1] for x in freq_train_list].count(1))

The most frequent words:
the: 336716
and: 163922
a: 162987
of: 145859
to: 135708
is: 107314
in: 93964
it: 79044
i: 77155
this: 75999
that: 69789

The least frequent words:
lowish: 1
hornes: 1
petunias: 1
inchworms: 1
billionare: 1
hightly: 1
slagged: 1
malkovitchesque: 1
muppified: 1
hued: 1

Words count: 80476

Count of word with frequency = 1: 31770


As expected the most frequent words are articles and prepositions.

But there is also a big fraction of words with the frequency of 1 despite the fact that there is quite big number of texts in training set. However, the absense of lemmatization should be taken into consideration.